# PolEmo2 — Pierwsze modele klasyfikacji tekstu

W kolejnych komórkach znajdziesz objaśnienia, kod i wyniki doświadczeń.
Czytaj je po kolei i uruchamiaj kod przyciskiem ▶ albo **Shift+Enter**.
Definicje funkcji przygotowują doświadczenie; obliczenia rozpocznie
dopiero komórka **Uruchomienie porównania** pod koniec notatnika.

Uczymy model rozpoznawać wydźwięk polskich opinii. W jednym doświadczeniu
używamy całego zbioru. W drugim uczymy model na tekstach o hotelach
i testujemy go na tekstach medycznych. Każdy model dopasowujemy osobno.

Wybierz **Plik → Zapisz kopię na Dysku**.
W tej części wystarczy CPU; nie potrzebujesz akceleratora GPU.
Materiały, dane i obliczenia nie wymagają płatnych usług ani klucza API.

[Artykuł i materiały](https://neuronovo.si/projekty/polemo2/).


In [ ]:
import importlib.util
import subprocess
import sys

# Colab udostępnia większość bibliotek. Instalujemy tylko brakujące,
# z oficjalnego rejestru PyPI; nie zmieniamy całego środowiska.
pakiety = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib'}
brakujace = [nazwa_pypi for modul, nazwa_pypi in pakiety.items()
             if importlib.util.find_spec(modul) is None]
if brakujace:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *brakujace])
print("Biblioteki są dostępne.")


## Biblioteki, źródła i nazwy metod

Poniżej widać importy i stałe: wersję danych, wersję modelu, źródło
polskich wektorów fastText oraz podziały obu doświadczeń. Nie pobieramy
najnowszej, zmieniającej się wersji modelu: wskazujemy konkretną rewizję.
`ORIGINAL_MODULE_SHA256` dokumentuje źródło tej implementacji.
Pole `module_sha256` w raporcie odnosi się do tego oryginalnego pliku,
z którego funkcje rozdzielono na komórki; nie jest sumą całego notatnika.


In [ ]:
from __future__ import annotations
import copy
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import sys
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler
from types import SimpleNamespace

ORIGINAL_MODULE_SHA256 = "44b4be5cbf87b2d85696d5e6c7bbb6109ced6816a687695c85296be7376ef015"

SEED = 42

DATASET_ID = "clarin-pl/polemo2-official"

DATASET_REVISION = "802e35d2b12bae84bb07911d841e8f046dc2fcef"

ENCODER_ID = "sdadas/polish-distilroberta"

ENCODER_REVISION = "849b664fa3134beae84095d28a184c145c6a3aa5"

FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pl.vec"

LABELS = ["zero", "minus", "plus", "amb"]

METHOD_NAMES = {
    "dummy": "Najczęstsza klasa",
    "tfidf_nb": "TF-IDF słów + naiwny klasyfikator Bayesa",
    "tfidf_lr": "TF-IDF słów + regresja logistyczna",
    "tfidf_word_char_lr": "TF-IDF słów i fragmentów słów + regresja logistyczna",
    "fasttext_lr": "Średnia wektorów fastText PL + regresja logistyczna",
    "encoder_frozen_lr": "Zamrożony polski encoder + regresja logistyczna",
    "encoder_finetuned": "Dostrajanie polskiego encodera",
}

CLASSICAL_METHODS = list(METHOD_NAMES)[:5]

ENCODER_METHODS = list(METHOD_NAMES)[5:]

PROTOCOLS = {
    "all_text": {
        "train": ("all", "train"),
        "validation": ("all", "dev"),
        "test": ("all", "test"),
    },
    "hotels_to_medicine": {
        "train": ("hotels", "train"),
        "validation": ("hotels", "dev"),
        "test": ("medicine", "test"),
    },
}


## Pobieranie plików i sumy kontrolne

Dane pobieramy porcjami na dysk. Cały duży plik fastText nie musi
jednocześnie zajmować pamięci RAM. Suma SHA256 pozwala sprawdzić,
czy dwa uruchomienia korzystały z tych samych plików.
Raport JSON zapisujemy atomowo, aby przerwana sesja nie uszkodziła
wcześniejszych wyników.


In [ ]:
def _json_write(path, value):
    """Zapis atomowy chroni poprzedni raport, gdy sesja zostanie przerwana."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)
    temporary.replace(path)

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _download(url, destination):
    """Pobieramy porcjami na dysk; cały duży plik nie trafia do pamięci RAM."""
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "Neuronovo-PolEmo2-education/1.0"})
    print(f"Pobieranie: {destination.name}", flush=True)
    try:
        with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as handle:
            shutil.copyfileobj(response, handle, length=1024 * 1024)
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    return destination


## Co porównujemy

| Doświadczenie | Zbiór uczący | Walidacja | Test |
|---|---:|---:|---:|
| `all_text`: wszystkie domeny | 6573 teksty | 823 teksty | 820 tekstów |
| `hotels_to_medicine`: hotele → medycyna | 3165 opinii o hotelach | 396 opinii o hotelach | 327 tekstów medycznych |

W drugim doświadczeniu oceniamy te same, już wybrane wagi także na
**395 hotelowych tekstach testowych** (`reference_test`). Możemy wtedy
porównać wynik dla opinii o hotelach z wynikiem dla tekstów medycznych.
Różnica wyników może zależeć również od udziałów klas i trudności tekstów.

Zachowujemy oficjalne podziały autora. Nie tworzymy nowego losowego podziału
i nie łączymy dokumentów `all_text` ze zdaniami `all_sentence`.
Każdy model dopasowujemy osobno w obu doświadczeniach. W doświadczeniu
hotelowym nie korzystamy z wag dostrojonych wcześniej w `all_text`.

Etykiety to `zero` (neutralna), `minus` (negatywna), `plus` (pozytywna)
i `amb` (mieszana/ambiwalentna). Ostatnia klasa obejmuje teksty z elementami
pozytywnymi i negatywnymi; nie muszą one występować w równych proporcjach.


In [ ]:
def load_split(domain, split, cache_dir="polemo2-cache"):
    """Czytamy format autora: tekst i etykieta w ostatnim tokenie wiersza.

    W danych występują także etykiety plus_m i minus_m. Zgodnie z loaderem
    autora sprowadzamy je do plus i minus. Nie mieszamy wariantu dokumentów
    (text) z wariantem pojedynczych zdań (sentence).
    """
    filename = f"{domain}.text.{split}.txt"
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/data/{filename}"
    path = _download(url, Path(cache_dir) / DATASET_REVISION / filename)
    texts, targets = [], []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.rsplit(maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Brak etykiety: {filename}, wiersz {line_number}.")
        text, raw_label = parts
        match = re.search(r"(?:^|_)(zero|minus|plus|amb)(?:_m)?$", raw_label)
        if match is None:
            raise ValueError(f"Nieznana etykieta w {filename}, wiersz {line_number}: {raw_label!r}.")
        texts.append(text)
        targets.append(LABELS.index(match.group(1)))
    if not texts:
        raise ValueError(f"Pusty zbiór: {filename}.")
    frame = pd.DataFrame({"text": texts, "target": targets})
    metadata = {"filename": filename, "rows": len(frame), "sha256": _sha256_file(path), "source": url}
    return frame, metadata


## Miary, sprzęt i czas

Główna miara to macro-F1: średnia z F1 policzonego osobno dla każdej
klasy. Każda z czterech klas ma taką samą wagę. Dodatkowo zapisujemy
trafność (`accuracy`) i średnią czułość klas (`balanced_accuracy`).
Wyniki mieszczą się w zakresie od 0 do 1.

W raporcie zapisujemy wersje bibliotek i nazwę sprzętu. Przed pomiarem
czasu czekamy na zakończenie działań GPU, bo są wykonywane asynchronicznie.
Na GPU używamy mieszanej precyzji FP16 (ang. automatic mixed precision),
a na CPU — FP32. Nie obiecujemy konkretnego czasu działania.


In [ ]:
def _versions():
    result = {"python": platform.python_version()}
    for name in ["numpy", "pandas", "scipy", "scikit-learn", "torch", "transformers", "tokenizers", "safetensors", "huggingface-hub"]:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result

def _hardware():
    result = {"platform": platform.platform(), "cpu": platform.processor() or platform.machine(), "cpu_count": os.cpu_count()}
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                result["cpu"] = line.split(":", 1)[1].strip()
                break
    except OSError:
        pass
    try:
        import torch
        result["cuda_available"] = bool(torch.cuda.is_available())
        result["cuda_version"] = torch.version.cuda
        if torch.cuda.is_available():
            prop = torch.cuda.get_device_properties(0)
            result["gpu"] = prop.name
            result["gpu_memory_gib"] = round(prop.total_memory / 1024**3, 2)
    except ImportError:
        result["cuda_available"] = False
    return result

def _synchronize():
    # Operacje GPU są asynchroniczne. Bez synchronizacji czas byłby zaniżony.
    if "torch" in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()

def _clock():
    _synchronize()
    return time.perf_counter()

def _amp_context(device):
    """Na GPU używamy mieszanego FP16; na CPU pozostajemy przy FP32."""
    import torch
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda")

def _scores(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


## Jak wybieramy parametry i mierzymy wynik

Zbiór **uczący** służy do wyznaczenia wag, **walidacja** do wyboru parametrów,
a **test** do końcowej oceny. Żaden klasyfikator, skaler ani słownik TF-IDF
nie jest dopasowywany do walidacji lub testu.

Główną miarą jest **macro-F1**: liczymy F1 osobno dla każdej z czterech klas,
a potem średnią. Każda klasa ma w tej średniej taką samą wagę. Zapisujemy
również odsetek poprawnych odpowiedzi (`accuracy`) i średnią czułość klas
(`balanced_accuracy`). Wszystkie wartości mieszczą się w zakresie od 0 do 1.

Dla Bayesa sprawdzamy `alpha = 0.1, 1, 10`, a dla regresji logistycznej
`C = 0.1, 1, 4`. Wygrywa najwyższe macro-F1 na walidacji; przy remisie
zostaje pierwszy parametr z tej kolejności. Najlepszą metodę także
wskazujemy według walidacji. Wynik testowy nie służy do podejmowania tych decyzji.

Plan zostaje zapisany w `protocol.json` przed oceną testową. W tym notatniku
nie zmieniaj siatki parametrów po obejrzeniu testu. Własne pomysły sprawdzaj
na treningu i walidacji; powtarzana ocena na tym samym teście przestaje
być niezależnym sprawdzianem.


In [ ]:
def _configuration():
    """Cały plan zapisujemy przed pierwszym odczytem zbioru testowego."""
    return {
        "primary_metric": "macro_f1", "seed": SEED,
        "selection": "Najwyższe macro-F1 walidacji; remis rozstrzyga kolejność siatki.",
        "tfidf": {"word_ngram_range": [1, 2], "min_df": 2, "sublinear_tf": True, "max_features": 60000},
        "char_tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5], "max_features": 60000},
        "naive_bayes_alpha_grid": [0.1, 1.0, 10.0],
        "logistic_C_grid": [0.1, 1.0, 4.0],
        "logistic_max_iter": 2000, "logistic_class_weight": "balanced",
        "encoder": {"model": ENCODER_ID, "revision": ENCODER_REVISION, "max_length": 256, "batch_size": 16, "learning_rate": 2e-5, "max_epochs": 8, "patience": 2, "min_delta": 1e-4, "weight_decay": 0.01, "restore_best": True, "training_precision": "CUDA: AMP FP16 + GradScaler; CPU: FP32", "inference_precision": "CUDA: AMP FP16; CPU: FP32"},
        "fasttext": {"source": FASTTEXT_URL, "vocabulary": "Unia słów z wejściowych tekstów, bez użycia ich etykiet; wyłącznie filtr pamięci gotowego słownika", "oov": "Pominięte; brak znanych słów daje wektor zerowy"},
        "test_usage": "Etykiety testu służą wyłącznie końcowej ocenie, po wyborze parametrów danej metody. Tekst testowy może wcześniej służyć tylko do wyszukania stałego, gotowego wektora fastText; nie uczymy na nim żadnych wag ani statystyk.",
    }


## Najczęstsza klasa i reprezentacje TF-IDF

Model odniesienia zawsze odpowiada najczęstszą klasą z treningu;
nie czyta tekstu. Kolejne metody zamieniają tekst na TF-IDF słów
i par kolejnych słów. Para „nie polecam” może wnosić inną informację
niż samo „polecam”. Słownik i wagi IDF wyznaczamy tylko na treningu.

Trzecia reprezentacja łączy słowa z fragmentami o długości 3–5 znaków
w obrębie słowa (`char_wb`). Mogą one pomagać przy odmianie wyrazów
i literówkach. Każda część ma limit 60 000 cech.
Regresja logistyczna jest tutaj klasyfikatorem czterech kategorii.


In [ ]:
def _word_tfidf():
    # Nie usuwamy częstych słów ani nie sprowadzamy słów do form podstawowych.
    # Słownik oraz wagi IDF powstają wyłącznie na zbiorze uczącym.
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=60000, dtype=np.float32)

def _features_for(method):
    if method == "tfidf_word_char_lr":
        return FeatureUnion([
            ("words", _word_tfidf()),
            ("characters", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=60000, dtype=np.float32)),
        ])
    return _word_tfidf()

def _prepare_tfidf(context, row, method):
    """Dopasuj słownik i IDF do treningu; walidację tylko przekształć."""
    vectorizer = _features_for(method)
    started = _clock()
    x_train = vectorizer.fit_transform(context.train_texts)
    row['feature_time'] = _clock() - started
    started = _clock()
    x_validation = vectorizer.transform(context.validation_texts)
    validation_feature_s = _clock() - started
    transform = vectorizer.transform
    return dict(x_train=x_train, x_validation=x_validation, validation_feature_s=validation_feature_s, transform=transform)

def _run_dummy(context, row):
    """Ucz punkt odniesienia bez czytania tekstu."""
    estimator = DummyClassifier(strategy='most_frequent')
    started = _clock()
    estimator.fit(np.zeros((len(context.train), 1)), context.train.target)
    row['fit_s'] = row['train_s'] = _clock() - started
    started = _clock()
    predicted = estimator.predict(np.zeros((len(context.validation), 1)))
    validation_s = _clock() - started
    row['validation'] = _scores(context.validation.target, predicted)
    row['selected_params'] = {'strategy': 'most_frequent'}
    current_test = _get_test(context)
    started = _clock()
    predicted = estimator.predict(np.zeros((len(current_test), 1)))
    test_s = _clock() - started
    _evaluate_reference(context, row, lambda texts: estimator.predict(np.zeros((len(texts), 1))))
    return (predicted, current_test, validation_s, test_s)


## Wybór klasyfikatora na walidacji

Dla naiwnego klasyfikatora Bayesa sprawdzamy `alpha = 0.1, 1, 10`.
Dla regresji logistycznej sprawdzamy `C = 0.1, 1, 4`; mniejsze `C`
mocniej ogranicza wielkość wag. Wygrywa najwyższe macro-F1 walidacji.
Przy remisie zostaje pierwszy parametr z zapisanej kolejności.
`class_weight="balanced"` zwiększa wagę przykładów z rzadszych klas.

Każdy kandydat uczy się wyłącznie na treningu. Dla gęstych wektorów
także skaler dopasowujemy tylko do treningu. Ta funkcja nie ma dostępu
do zbioru testowego.


In [ ]:
def _linear_model(c):
    return LogisticRegression(C=c, max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=SEED)

def _select_classifier(method, x_train, y_train, x_validation, y_validation, dense=False):
    """Parametr wybiera macro-F1 walidacji. Przy remisie wygrywa pierwszy.

    Każdy kandydat uczy się od początku na treningu. StandardScaler także
    dopasowujemy wyłącznie na treningu, nigdy na walidacji ani teście.
    """
    values = [0.1, 1.0, 10.0] if method == "tfidf_nb" else [0.1, 1.0, 4.0]
    best, best_score, history, fit_s, validation_s = None, -1.0, [], 0.0, 0.0
    for value in values:
        estimator = MultinomialNB(alpha=value) if method == "tfidf_nb" else _linear_model(value)
        if dense:
            estimator = make_pipeline(StandardScaler(), estimator)
        started = _clock()
        estimator.fit(x_train, y_train)
        fit_s += _clock() - started
        started = _clock()
        predicted = estimator.predict(x_validation)
        validation_s += _clock() - started
        metrics = _scores(y_validation, predicted)
        params = {"alpha" if method == "tfidf_nb" else "C": value}
        history.append({"params": params, **metrics})
        print(f"  {params}: walidacja macro-F1={metrics['macro_f1']:.4f}", flush=True)
        if metrics["macro_f1"] > best_score + 1e-12:
            best_score = metrics["macro_f1"]
            best = (estimator, predicted, metrics, params)
    return (*best, history, fit_s, validation_s)


## Polskie wektory fastText

Każde znane słowo otrzymuje gotowy wektor 300 liczb, nauczony
wcześniej na polskiej Wikipedii. Tekst opisujemy średnią tych wektorów.
Średnia nie zachowuje kolejności słów.

Oficjalny plik `.vec` czytamy wierszami. Zachowujemy wektory słów
potrzebnych do opisania wejściowych tekstów, także walidacji i testu.
To wyłącznie oszczędność RAM: wektory są stałe, a filtr nie używa
etykiet. Daje te same reprezentacje co cały gotowy słownik.
Format `.vec` nie oblicza wektorów nowych słów z ich fragmentów.
Nieznane słowa pomijamy; tekst bez znanych słów ma wektor zerowy.
Raport podaje pokrycie słownika.


In [ ]:
def _tokens(text):
    # Taka sama prosta tokenizacja w treningu, walidacji i teście.
    return re.findall(r"(?u)\b\w+\b", text.lower())

def _load_fasttext(input_texts, cache_dir):
    """Wczytujemy tylko potrzebną część gotowego, stałego słownika.

    Plik .vec zawiera gotowe wektory słownikowe. Nie jest modelem .bin,
    więc nie potrafi wyliczyć wektora nowego słowa z jego fragmentów.
    Filtrowanie po słowach z wejściowych tekstów oszczędza RAM i daje te same
    wektory co wczytanie całego pliku. Nie używa etykiet ani nie uczy wag.
    Słowa spoza zachowanego słownika pomijamy; pusty tekst ma wektor zerowy.
    """
    wanted = {word for text in input_texts for word in _tokens(text)}
    path = _download(FASTTEXT_URL, Path(cache_dir) / "wiki.pl.vec")
    vectors = {}
    with path.open("rb") as handle:
        header = handle.readline().split()
        if len(header) != 2:
            raise ValueError("Niepoprawny nagłówek pliku fastText.")
        dimension = int(header[1])
        for line in handle:
            raw_word, separator, raw_values = line.rstrip().partition(b" ")
            word = raw_word.decode("utf-8", errors="replace")
            if separator and word in wanted:
                vector = np.fromstring(raw_values.decode("ascii"), sep=" ", dtype=np.float32)
                if vector.shape == (dimension,):
                    vectors[word] = vector
    return vectors, dimension, {
        "source": FASTTEXT_URL,
        "sha256": _sha256_file(path),
        "dimension": dimension,
        "lookup_vocabulary": len(wanted),
        "retained_vectors": len(vectors),
        "license": "CC-BY-SA-3.0",
    }

def _average_vectors(texts, vectors, dimension):
    result = np.zeros((len(texts), dimension), dtype=np.float32)
    found = total = empty = 0
    for index, text in enumerate(texts):
        words = _tokens(text)
        total += len(words)
        available = [vectors[word] for word in words if word in vectors]
        found += len(available)
        if available:
            result[index] = np.mean(available, axis=0)
        else:
            empty += 1
    return result, {"token_coverage": found / total if total else 0.0, "zero_vectors": empty}

def _prepare_fasttext(context, row):
    """Wyszukaj stałe wektory; nie dopasowuj ich do etykiet walidacji ani testu."""
    started = _clock()
    lookup_texts = context.train_texts + context.validation_texts + _get_test(context).text.tolist()
    reference = _get_reference_test(context)
    if reference is not None:
        lookup_texts += reference.text.tolist()
    vectors, dimension, ft_meta = _load_fasttext(lookup_texts, context.cache_dir)
    row['setup_s'] = _clock() - started
    row['fasttext'] = ft_meta
    started = _clock()
    x_train, training_coverage = _average_vectors(context.train_texts, vectors, dimension)
    row['feature_time'] = _clock() - started
    started = _clock()
    x_validation, validation_coverage = _average_vectors(context.validation_texts, vectors, dimension)
    validation_feature_s = _clock() - started
    row['coverage'] = {'train': training_coverage, 'validation': validation_coverage}
    transform = lambda texts: _average_vectors(texts, vectors, dimension)[0]
    row['notes'].append('Stały słownik .vec filtrujemy po słowach z wejściowych tekstów wyłącznie dla oszczędności RAM. Etykiety walidacji i testu nie wpływają na wektory. Brak wektora w oryginalnym pliku oznacza pominięcie słowa.')
    return dict(x_train=x_train, x_validation=x_validation, validation_feature_s=validation_feature_s, transform=transform, vectors=vectors, dimension=dimension)


## Dopasowanie klasyfikatora i końcowa predykcja

Łączymy przygotowaną reprezentację ze wspólną funkcją wyboru parametru.
Zbiór testowy trafia wyłącznie do wybranego modelu. Hotelowy test
odniesienia oceniamy tymi samymi wagami; nie dopasowujemy na nim niczego.
W każdym doświadczeniu przygotowanie reprezentacji i uczenie zaczynają
się osobno, bez przenoszenia dostrojonych wag między doświadczeniami.


In [ ]:
def _run_vector_method(context, row, method):
    """Wybierz klasyfikator po walidacji; potem oceń wybraną konfigurację."""
    if method == "fasttext_lr":
        features = _prepare_fasttext(context, row)
    elif method == "encoder_frozen_lr":
        features = _prepare_frozen(context, row)
    else:
        features = _prepare_tfidf(context, row, method)
    dense = method in {"fasttext_lr", "encoder_frozen_lr"}
    estimator, _, row["validation"], row["selected_params"], row["validation_history"], row["fit_s"], selection_predict_s = _select_classifier(
        method, features["x_train"], context.train.target,
        features["x_validation"], context.validation.target, dense=dense,
    )
    row["train_s"] = row["feature_time"] + row["fit_s"]
    validation_s = features["validation_feature_s"] + selection_predict_s
    current_test = _get_test(context)
    started = _clock()
    predicted = estimator.predict(features["transform"](current_test.text.tolist()))
    test_s = _clock() - started
    _evaluate_reference(context, row, lambda texts: estimator.predict(features["transform"](texts)))
    if method == "fasttext_lr":
        _, coverage = _average_vectors(current_test.text.tolist(), features["vectors"], features["dimension"])
        row["coverage"]["test"] = coverage
    if method == "encoder_frozen_lr":
        row["truncation_rate"]["test"] = _truncation_rate(current_test.text.tolist(), features["tokenizer"])
    return predicted, current_test, validation_s, test_s


## Zapis planu i przygotowanie doświadczenia

Tworzymy katalog wyników, zapisujemy konfigurację i wczytujemy trening
oraz walidację. Przy ponownym uruchomieniu wybranych metod można zachować
inne, wcześniej zapisane wyniki tego samego planu. Każdy wiersz zachowuje
własne informacje o sprzęcie i wersjach bibliotek.


In [ ]:
def _create_context(protocol, output_dir, methods, cache_dir):
    """Zapisz plan i wczytaj wyłącznie trening oraz walidację."""
    'Uruchom jeden protokół; zapisuj raport po zakończeniu każdej metody.\n\n    Parametry methods mogą wskazać np. CLASSICAL_METHODS lub ENCODER_METHODS.\n    Przy ponownym uruchomieniu tego samego protokołu zachowujemy wcześniejsze\n    wiersze raportu, a wskazane metody mierzymy od początku.\n    '
    if protocol not in PROTOCOLS:
        raise ValueError(f'Nieznany protokół: {protocol}. Dostępne: {list(PROTOCOLS)}')
    methods = list(METHOD_NAMES) if methods is None else list(methods)
    if not methods or len(set(methods)) != len(methods) or any((method not in METHOD_NAMES for method in methods)):
        raise ValueError('Podaj niepustą listę różnych identyfikatorów z METHOD_NAMES.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    specification = json.loads(json.dumps({'name': protocol, 'splits': PROTOCOLS[protocol], **_configuration()}))
    _json_write(output_dir / 'protocol.json', specification)
    data_started = _clock()
    train, train_meta = load_split(*PROTOCOLS[protocol]['train'], cache_dir)
    validation, validation_meta = load_split(*PROTOCOLS[protocol]['validation'], cache_dir)
    data_setup_s = _clock() - data_started
    for name, frame in [('train', train), ('validation', validation)]:
        if set(frame.target) != set(range(len(LABELS))):
            raise ValueError(f'Zbiór {name} nie zawiera wszystkich czterech klas.')
    report = {'status': 'running', 'run': {'date': datetime.now(timezone.utc).isoformat(), 'protocol': protocol, 'hardware': _hardware(), 'versions': _versions(), 'protocol_config': specification, 'module_sha256': ORIGINAL_MODULE_SHA256, 'data_setup_s': data_setup_s, 'data': {'dataset': DATASET_ID, 'revision': DATASET_REVISION, 'variant': 'all_text' if protocol == 'all_text' else 'hotels_text → medicine_text', 'train_rows': len(train), 'validation_rows': len(validation), 'sources': {'train': train_meta, 'validation': validation_meta}, 'labels': LABELS}, 'notes': ['Wyniki pojedynczego uruchomienia; nie są średnią z wielu losowych inicjalizacji.', 'Teksty, etykiety poszczególnych przykładów i identyfikatory wierszy nie są eksportowane.']}, 'results': []}
    existing_path = output_dir / 'results.json'
    if existing_path.exists():
        previous = json.loads(existing_path.read_text(encoding='utf-8'))
        if previous.get('run', {}).get('protocol_config') == specification and previous.get('run', {}).get('data', {}).get('revision') == DATASET_REVISION:
            report['results'] = [row for row in previous.get('results', []) if row['id'] not in methods]
    _save_report(report, output_dir)
    train_texts, validation_texts = (train.text.tolist(), validation.text.tolist())
    return SimpleNamespace(protocol=protocol, output_dir=output_dir, cache_dir=cache_dir, methods=methods, train=train, validation=validation, train_texts=train_texts, validation_texts=validation_texts, report=report, test=None, test_meta=None, reference_test=None)


## Dwa testy w doświadczeniu ze zmianą domeny

Końcowa ocena korzysta z oficjalnego testu. W doświadczeniu hotelowym
główny test obejmuje 327 tekstów medycznych, a `reference_test` —
395 tekstów o hotelach odłożonych do testu. Żaden nie służy do strojenia.
`domain_gap` oznacza wynik hotelowy minus wynik medyczny. Różnica może
zależeć także od udziałów klas i trudności tekstów.


In [ ]:
def _get_test(context):
    """Ocena odłożonych danych bez dopasowywania wag."""
    if context.test is None:
        context.test, context.test_meta = load_split(*PROTOCOLS[context.protocol]['test'], context.cache_dir)
        context.report['run']['data']['test_rows'] = len(context.test)
        context.report['run']['data']['sources']['test'] = context.test_meta
    return context.test

def _get_reference_test(context):
    """Ocena odłożonych danych bez dopasowywania wag."""
    if context.protocol != 'hotels_to_medicine':
        return None
    if context.reference_test is None:
        context.reference_test, metadata = load_split('hotels', 'test', context.cache_dir)
        context.report['run']['data']['reference_test_rows'] = len(context.reference_test)
        context.report['run']['data']['sources']['reference_test'] = metadata
    return context.reference_test

def _evaluate_reference(context, row, predict_function):
    """Ocena odłożonych danych bez dopasowywania wag."""
    reference = _get_reference_test(context)
    if reference is not None:
        started = _clock()
        predictions = predict_function(reference.text.tolist())
        row['reference_test_predict_s'] = _clock() - started
        row['reference_test'] = _scores(reference.target, predictions)


## Co znajdziesz w plikach wynikowych

Raport zapisuje wynik po każdej metodzie, także wtedy, gdy inna metoda
zakończy się błędem. Błąd ma status `error`; nie zastępujemy go liczbą.
Pełne historie walidacji, wybrane parametry i wyniki hotelowego testu
odniesienia są w JSON. CSV zawiera główne miary walidacji i testu.

`train_s` obejmuje przygotowanie reprezentacji treningowych oraz uczenie
wszystkich kandydatów sprawdzanych w siatce, a dla dostrajania — wszystkie
wykonane epoki. Nie jest to wyłącznie czas uczenia zwycięzcy.
`predict_s.validation` obejmuje ocenę kandydatów lub kolejnych epok.
`predict_s.test` mierzy pełną predykcję wybranego modelu, razem z zamianą
tekstów na wektory. Pobranie i wczytanie gotowych wag znajduje się osobno
w `setup_s`. Nie porównuj samych liczb sekund bez sprawdzenia sprzętu.

Raport zawiera wersje bibliotek, nazwę GPU/CPU, rewizje danych i modelu
oraz sumy kontrolne źródeł. Nie zawiera tekstów ani etykiet poszczególnych
przykładów. Pojedyncze uruchomienie nie pokazuje całej zmienności wyniku
wynikającej z losowej inicjalizacji i pracy GPU.


In [ ]:
def _save_report(report, output_dir):
    output_dir = Path(output_dir)
    _json_write(output_dir / "results.json", report)
    rows = []
    for row in report["results"]:
        flat = {key: row.get(key) for key in ["id", "name", "status", "train_s", "setup_s", "feature_time", "fit_s", "best_epoch"]}
        for split in ["validation", "test"]:
            for metric, value in row.get(split, {}).items():
                flat[f"{split}_{metric}"] = value
            flat[f"{split}_predict_s"] = row.get("predict_s", {}).get(split)
        flat["selected_params"] = json.dumps(row.get("selected_params", {}), ensure_ascii=False)
        rows.append(flat)
    if rows:
        pd.DataFrame(rows).to_csv(output_dir / "results.csv", index=False)

def _compact_report(report):
    """Wydruk mieści się w Colabie; pełne historie są w pliku JSON."""
    compact = copy.deepcopy(report)
    if "protocols" in compact:
        compact["protocols"] = {key: _compact_report(value) for key, value in compact["protocols"].items()}
    else:
        for row in compact["results"]:
            row.pop("validation_history", None)
    return compact

def _run_one_method(context, method):
    """Wykonaj metodę, zapisz prawdziwy wynik albo jawny błąd i zwolnij pamięć."""
    print(f'\n[{context.protocol}] {METHOD_NAMES[method]}', flush=True)
    row = {'id': method, 'name': METHOD_NAMES[method], 'status': 'running', 'setup_s': 0.0, 'feature_time': 0.0, 'fit_s': 0.0, 'truncation_rate': None, 'notes': [], 'selected_params': {}, 'validation_history': []}
    row['run_context'] = {'date': datetime.now(timezone.utc).isoformat(), 'hardware': context.report['run']['hardware'], 'versions': context.report['run']['versions'], 'module_sha256': context.report['run']['module_sha256']}
    try:
        if method == 'dummy':
            predicted, current_test, validation_s, test_s = _run_dummy(context, row)
        elif method in {'tfidf_nb', 'tfidf_lr', 'tfidf_word_char_lr', 'fasttext_lr', 'encoder_frozen_lr'}:
            predicted, current_test, validation_s, test_s = _run_vector_method(context, row, method)
        else:
            predicted, current_test, validation_s, test_s = _run_finetuned(context, row)
        row['test'] = _scores(current_test.target, predicted)
        if 'reference_test' in row:
            row['domain_gap'] = {key: row['reference_test'][key] - row['test'][key] for key in row['test']}
        row['predict_s'] = {'validation': validation_s, 'test': test_s}
        row['status'] = 'completed'
        print(f"  Wynik: val macro-F1={row['validation']['macro_f1']:.4f}; test macro-F1={row['test']['macro_f1']:.4f}", flush=True)
    except Exception as error:
        row['status'] = 'error'
        row['error'] = f'{type(error).__name__}: {error}'
        print(f"  Błąd: {row['error']}", flush=True)
    context.report['results'].append(row)
    context.report['results'].sort(key=lambda result: list(METHOD_NAMES).index(result['id']))
    _save_report(context.report, context.output_dir)
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## Wykonanie planu i archiwum

Te funkcje wywołują opisane wyżej metody, łączą ich raporty i tworzą ZIP.
Najlepszą metodę wybierają według walidacji. Jeśli obliczenie zakończy się
błędem, w raporcie zostaje jawny status i komunikat. Nie wpisujemy
zastępczych ani przewidywanych wyników.


In [ ]:
def run_protocol(protocol="all_text", output_dir="polemo2-results", methods=None, cache_dir="polemo2-cache"):
    """Uruchom wybrane metody według jawnego planu jednego doświadczenia."""
    context = _create_context(protocol, output_dir, methods, cache_dir)
    for method in context.methods:
        _run_one_method(context, method)
    complete_rows = [row for row in context.report['results'] if row['status'] == 'completed']
    if complete_rows:
        best = max(complete_rows, key=lambda row: row['validation']['macro_f1'])
        context.report['run']['best_method_by_validation'] = best['id']
    context.report['status'] = 'completed' if all((row['status'] == 'completed' for row in context.report['results'])) else 'partial'
    _save_report(context.report, context.output_dir)
    return context.report

def run_all(output_dir="polemo2-results", methods=None, protocols=None, cache_dir="polemo2-cache"):
    """Uruchom oba doświadczenia i zapisz JSON, CSV oraz archiwum ZIP.

    Przykład krótkiego uruchomienia: run_all(methods=CLASSICAL_METHODS).
    Kolejne run_all(methods=ENCODER_METHODS) dopisze pozostałe metody do tych
    samych raportów, o ile konfiguracja i rewizja danych są takie same.
    """
    protocols = list(PROTOCOLS) if protocols is None else list(protocols)
    if not protocols or any(protocol not in PROTOCOLS for protocol in protocols):
        raise ValueError(f"Dostępne protokoły: {list(PROTOCOLS)}")
    output_dir = Path(output_dir)
    combined = {"status": "running", "protocols": {}}
    for protocol in protocols:
        combined["protocols"][protocol] = run_protocol(protocol, output_dir / protocol, methods=methods, cache_dir=cache_dir)
        combined["status"] = "completed" if all(report["status"] == "completed" for report in combined["protocols"].values()) else "partial"
        _json_write(output_dir / "results.json", combined)
    csv_frames = []
    for protocol in combined["protocols"]:
        csv_path = output_dir / protocol / "results.csv"
        if csv_path.exists():
            frame = pd.read_csv(csv_path)
            frame.insert(0, "protocol", protocol)
            csv_frames.append(frame)
    if csv_frames:
        pd.concat(csv_frames, ignore_index=True).to_csv(output_dir / "results.csv", index=False)
    archive = shutil.make_archive(str(output_dir.resolve()), "zip", root_dir=output_dir)
    print("\nNEURONOVO_RESULTS_JSON", flush=True)
    print(json.dumps(_compact_report(combined), ensure_ascii=False, allow_nan=False), flush=True)
    print(f"Archiwum wyników: {archive}", flush=True)
    return combined


## Uruchomienie porównania

Teraz zaczynają się obliczenia. Porównujemy pięć metod w dwóch doświadczeniach. To 10 uruchomień metod.
Tam, gdzie przewidziano wybór parametrów, wykonujemy również kolejne dopasowania.
Postęp i wyniki walidacji pojawiają się poniżej. Zaczekaj na zakończenie
komórki. Jeśli sesja zostanie przerwana, zachowaj dostępne pliki wyników.


In [ ]:
raport = run_all(output_dir="polemo2-results", methods=CLASSICAL_METHODS)


## Zestawienie wyników

Walidacja i test znajdują się w osobnych kolumnach. Dla doświadczenia
hotelowego zobaczysz także wynik testu hotelowego. Najpierw sprawdź,
którą metodę wybralibyśmy według walidacji, a potem odczytaj jej wynik
testowy. Nie zakładamy z góry, która metoda okaże się najlepsza.


### Wyniki opublikowanego uruchomienia

Poniższe liczby pochodzą z zapisanego raportu. Nie są wynikami Twojej bieżącej sesji. Uruchomienie `run_all` tworzy własny raport.

**`all_text`** — data pomiaru: `2026-10-26T20:09:34.693832+00:00`; GPU: Tesla T4.

| Metoda | Macro-F1 walidacji | Macro-F1 testu | Trafność testu |
|---|---:|---:|---:|
| Najczęstsza klasa | 0.1349 | 0.1462 | 0.4134 |
| TF-IDF słów + naiwny klasyfikator Bayesa | 0.7921 | 0.8119 | 0.8244 |
| TF-IDF słów + regresja logistyczna | 0.8127 | 0.8070 | 0.8171 |
| TF-IDF słów i fragmentów słów + regresja logistyczna | 0.8276 | 0.8116 | 0.8159 |
| Średnia wektorów fastText PL + regresja logistyczna | 0.7683 | 0.7622 | 0.7646 |
| Zamrożony polski encoder + regresja logistyczna | 0.7813 | 0.7600 | 0.7610 |
| Dostrajanie polskiego encodera | 0.8431 | 0.8366 | 0.8451 |

**`hotels_to_medicine`** — data pomiaru: `2026-10-26T20:19:01.842494+00:00`; GPU: Tesla T4.

| Metoda | Macro-F1 walidacji | Macro-F1 testu | Trafność testu |
|---|---:|---:|---:|
| Najczęstsza klasa | 0.1387 | 0.1446 | 0.4067 |
| TF-IDF słów + naiwny klasyfikator Bayesa | 0.7787 | 0.4865 | 0.6239 |
| TF-IDF słów + regresja logistyczna | 0.8107 | 0.5117 | 0.6636 |
| TF-IDF słów i fragmentów słów + regresja logistyczna | 0.8270 | 0.5585 | 0.6881 |
| Średnia wektorów fastText PL + regresja logistyczna | 0.7965 | 0.5834 | 0.7064 |
| Zamrożony polski encoder + regresja logistyczna | 0.7934 | 0.6012 | 0.7064 |
| Dostrajanie polskiego encodera | 0.8128 | 0.6448 | 0.7248 |

[Pełny raport JSON z metadanymi](https://neuronovo.si/assets/projects/polemo2/results.json).


In [ ]:
import pandas as pd
from IPython.display import display

def pokaz_wyniki(raport):
    # Zestawiamy walidację i test w osobnych kolumnach.
    wiersze = []
    for protokol, czesc in raport["protocols"].items():
        for wynik in czesc["results"]:
            wiersz = {
                "doświadczenie": protokol,
                "metoda": wynik["name"],
                "status": wynik["status"],
                "F1 walidacji": wynik.get("validation", {}).get("macro_f1"),
                "F1 testu": wynik.get("test", {}).get("macro_f1"),
                "trafność testu": wynik.get("test", {}).get("accuracy"),
                "średnia czułość testu": wynik.get("test", {}).get("balanced_accuracy"),
                "uczenie [s]": wynik.get("train_s"),
            }
            if "reference_test" in wynik:
                wiersz["F1 testu hotelowego"] = wynik["reference_test"]["macro_f1"]
            if "error" in wynik:
                wiersz["błąd"] = wynik["error"]
            wiersze.append(wiersz)
    display(pd.DataFrame(wiersze).round(4))


# Własny pomiar ma pierwszeństwo. Bez niego można obejrzeć jawnie
# oznaczony raport opublikowany razem z artykułem.
if "raport" in globals():
    raport_do_podgladu = raport
    print("Wyniki obliczone w tej sesji notatnika.")
else:
    import json
    import urllib.request
    adres_wynikow = "https://neuronovo.si/assets/projects/polemo2/results.json"
    with urllib.request.urlopen(adres_wynikow) as odpowiedz:
        raport_do_podgladu = json.load(odpowiedz)
    print("Podgląd opublikowanego uruchomienia. Nie są to wyniki obliczone w tej sesji.")
pokaz_wyniki(raport_do_podgladu)


## Ćwiczenie: które klasy myli model

Sama średnia nie pokazuje wszystkich błędów. Obejrzymy macierz
pomyłek i kilka błędnie sklasyfikowanych tekstów **z walidacji**.
Odtwarzamy TF-IDF i regresję logistyczną dla `all_text`, używając `C`
wybranego już w raporcie na podstawie walidacji. Model ponownie
dopasowujemy wyłącznie do treningu.

To osobne ćwiczenie. Nie odczytuje testu, nie zmienia raportu
porównania i nie służy do wybierania parametrów po wyniku testowym.
W macierzy wiersze oznaczają klasy prawidłowe, a kolumny przewidywane.
Liczby na przekątnej to poprawne odpowiedzi.


In [ ]:
if "raport" not in globals():
    raise RuntimeError("Najpierw uruchom własne porównanie; podgląd opublikowanych wyników go nie zastępuje.")

import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

wynik_lr = next((w for w in raport["protocols"]["all_text"]["results"]
                if w["id"] == "tfidf_lr" and w["status"] == "completed"), None)
if wynik_lr is None:
    raise RuntimeError("Najpierw uruchom porównanie i sprawdź, czy TF-IDF + LR zakończył się poprawnie.")
wybrane_c = wynik_lr["selected_params"]["C"]
trening, _ = load_split("all", "train")
walidacja, _ = load_split("all", "dev")
model_do_analizy = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                    max_features=60000, dtype=np.float32),
    LogisticRegression(C=wybrane_c, max_iter=2000, class_weight="balanced",
                       solver="lbfgs", random_state=42),
)
model_do_analizy.fit(trening.text, trening.target)
przewidywania = model_do_analizy.predict(walidacja.text)
print("C wybrane wcześniej na walidacji:", wybrane_c)


import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    walidacja.target, przewidywania, labels=range(4),
    display_labels=LABELS, cmap="Blues",
)
plt.title("Macierz pomyłek — walidacja all_text, TF-IDF + LR")
plt.show()

# Oglądamy początek pięciu błędnie sklasyfikowanych opinii.
bledy = walidacja.loc[walidacja.target != przewidywania].copy()
bledy["prawidłowa klasa"] = bledy.target.map(dict(enumerate(LABELS)))
bledy["przewidziana klasa"] = [LABELS[i] for i in przewidywania[walidacja.target != przewidywania]]
bledy["tekst"] = bledy.text.str.slice(0, 500)
display(bledy[["tekst", "prawidłowa klasa", "przewidziana klasa"]].head(5).reset_index(drop=True))


## Pobierz wyniki

Archiwum zawiera raport zbiorczy i osobne raporty doświadczeń. Pobierz
je przed odłączeniem środowiska Colab. Raporty nie zawierają tekstów
poszczególnych opinii; przykłady z ćwiczenia pozostają tylko w jego wyniku.


In [ ]:
from google.colab import files

# Archiwum zawiera raport zbiorczy i osobne raporty obu doświadczeń.
# Pobierz je przed odłączeniem maszyny Colab.
files.download("polemo2-results.zip")


## Źródła

- [PolEmo2: oficjalne repozytorium danych](https://huggingface.co/datasets/clarin-pl/polemo2-official).
- [Kocoń, Miłkowski, Zaśko-Zielińska: opis zbioru PolEmo2](https://aclanthology.org/K19-1092/).
- [fastText: wektory Wikipedii, w tym języka polskiego](https://fasttext.cc/docs/en/pretrained-vectors.html) — CC BY-SA 3.0.
- [sdadas/polish-distilroberta: karta modelu i pliki](https://huggingface.co/sdadas/polish-distilroberta) — Apache 2.0.
- [scikit-learn: TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
  [regresja logistyczna](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html),
  [macro-F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).
- [Google Colab: zasady bezpłatnych zasobów](https://research.google.com/colaboratory/faq.html).

Kod używa dokładnie wskazanych wersji zbioru i modelu (ang. revision),
zapisanych w raporcie.
Szczegółowe warunki korzystania z danych znajdziesz w repozytorium PolEmo2.
